In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import h5py, os, tqdm, glob, scipy
# os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.49'
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

import jax
import jax.numpy as jnp
import jax_cosmo as jc

import optax
from optax.losses import huber_loss
from flax import nnx
import orbax.checkpoint as ocp
import jraph

import diffrax
from diffrax import diffeqsolve, ODETerm, LeapfrogMidpoint, PIDController, SaveAt, ConstantStepSize

import jaxpm
from jaxpm.painting import cic_paint, cic_read, compensate_cic
from jaxpm.kernels import fftk, gradient_kernel, invlaplace_kernel, longrange_kernel, invnabla_kernel
from jaxpm.utils import power_spectrum, cross_correlation_coefficients
from jaxpm.nn import MLP, CNN, HybridNet, AttentionGNN, ScaleConditionedCNN
from jaxpm import camels, plotting, hpm, nn, graph, data, diagnostics

# print(jax.devices("gpu"))
print(jax.default_backend())

gpu


# configuration

In [3]:
parts_per_dim = 64
mesh_per_dim = parts_per_dim
# mesh_per_dim = 2 * parts_per_dim
mesh_shape = [mesh_per_dim] * 3
box_size = [float(mesh_per_dim)] * 3

# CAMELS

In [4]:
train_dict = camels.load_CV_snapshots(
    "CV_0",
    mesh_per_dim,
    parts_per_dim,
    return_hydro=True,
)

cosmo = train_dict["cosmo"]
scales = train_dict["scales"]

dm_poss = train_dict["dm_poss"]
dm_vels = train_dict["dm_vels"]

gas_poss = train_dict["gas_poss"]
gas_vels = train_dict["gas_vels"]

Loaded /cluster/work/refregier/athomsen/flatiron/CAMELS/h5/SIMBA/CV/CV_0/parts=64,mesh=64.h5


In [5]:
dt0 = 0.01

print(np.diff(scales))
assert np.all(np.diff(scales) > dt0)

[0.02380952 0.03333333 0.02222222 0.02777778 0.01224106 0.01284044
 0.01346916 0.01412867 0.01482047 0.01554614 0.01630734 0.01710582
 0.0179434  0.01882198 0.01974359 0.02071031 0.02172438 0.0227881
 0.0239039  0.02507434 0.02630208 0.02758995 0.02894087 0.03035793
 0.03184439 0.03340362 0.03503921 0.03675488 0.03855455 0.04044235
 0.04242258 0.04449977 0.04667867]


In [6]:
# vali_dict = camels.load_CV_snapshots(
#     "CV_1",
#     mesh_per_dim,
#     parts_per_dim,
#     i_snapshots=i_snapshots,
#     return_hydro=True,
# )

In [7]:
vcic_paint = jax.vmap(cic_paint, in_axes=(None,0,None))
vcic_read = jax.vmap(cic_read, in_axes=(0,0))

In [8]:
@nnx.jit
def train_step(model, optimizer, y0, ts, ref):
    """JIT-compatible training step with dynamic snapshot range"""
    
    def loss_fn_wrapper(model):
        return field_loss_fn(model, y0, ts, ref, "cnn", eps=1e-8)
        
    loss, grads = nnx.value_and_grad(loss_fn_wrapper)(model)
    optimizer.update(grads)

    squared_sum = jax.tree_util.tree_reduce(
            lambda x, y: x + jnp.sum(y**2),
            grads,
            0.0
        )
    grad_norm = jnp.sqrt(squared_sum)
    
    return loss, grad_norm

In [9]:
@nnx.jit
def batched_train_step(model, optimizer, y0_batch, ts_batch, ref_batch):
    """Training step with batched inputs"""
    
    def loss_fn_wrapper(model):
        return batched_field_loss_fn(model, y0_batch, ts_batch, ref_batch, "cnn", eps=1e-8)
        
    loss, grads = nnx.value_and_grad(loss_fn_wrapper)(model)
    optimizer = optimizer.update(grads)

    squared_sum = jax.tree_util.tree_reduce(
            lambda x, y: x + jnp.sum(y**2),
            grads,
            0.0
        )
    grad_norm = jnp.sqrt(squared_sum)
    
    return loss, grad_norm

In [10]:
def solve_ode_diffrax(model, architecture, training=False):   
    ode = ODETerm(
        hpm.get_hpm_network_ode_fn(
            mesh_per_dim, 
            cosmo, 
            gravity_model=None, 
            pressure_model=model, 
            gas_architecture=architecture, 
            training=training,
        )
    )

    res = diffeqsolve(
            terms=ode,
            solver=LeapfrogMidpoint(),
            t0=scales[0],
            t1=scales[-1],
            dt0=dt0,
            y0=(dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]),
            saveat=SaveAt(ts=scales),
            max_steps=100,
            stepsize_controller=ConstantStepSize(),
        )
    res = res.ys

    return res

def solve_ode_sub(model, y0, ts, architecture, training=True):
    ode = ODETerm(
        hpm.get_hpm_network_ode_fn(
            mesh_per_dim, 
            cosmo, 
            gravity_model=None, 
            pressure_model=model, 
            gas_architecture=architecture, 
            training=training,
        )
    )

    res = diffeqsolve(
            terms=ode,
            solver=LeapfrogMidpoint(),
            t0=ts[0],
            t1=ts[-1],
            dt0=dt0,
            y0=y0,
            saveat=SaveAt(ts=ts),
            max_steps=100,
            stepsize_controller=ConstantStepSize(),
        )
    res = res.ys

    return res


# loss

### CAMELS ground truth

In [11]:
# per-particle reference
ref_pos = jnp.stack(gas_poss, axis=0)
ref_vel = jnp.stack(gas_vels, axis=0)

# field-level reference
gas_mass = cosmo.Omega_b / (cosmo.Omega_b + cosmo.Omega_c)
ref_rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)

# power spectrum reference
vpower_spectrum = jax.vmap(
    lambda fields: 
        power_spectrum(
            compensate_cic(fields),
            boxsize=np.array([25.0] * 3),
            kmin=np.pi / 25.0,
            dk=2 * np.pi / 25.0,
        )
)
_, ref_cls = vpower_spectrum(ref_rho)

vcross_correlation_separate = jax.vmap(
    lambda field_a, field_b:
        cross_correlation_coefficients(
            compensate_cic(field_a),
            compensate_cic(field_b),
            boxsize=np.array([25.0] * 3),
            kmin=np.pi / 25.0,
            dk=2 * np.pi / 25.0,
        )
)
vcross_correlation = lambda rhos: vcross_correlation_separate(rhos, ref_rho)

### particle-level

In [12]:
# def particle_loss_fn(model, architecture, huber=False, pos_dead_zone=False):
#     res = solve_ode_diffrax(model, architecture, training=True)
#     gas_poss = res[2]
#     gas_vels = res[3]

#     delta_pos = ((gas_poss - ref_pos + mesh_per_dim // 2) % mesh_per_dim) - mesh_per_dim // 2
#     pos_loss = jnp.sum(huber_loss(delta_pos), axis=-1)
    
#     pos_loss = jnp.mean(pos_loss)
    
#     res_rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)
#     kbins, res_cls = vpower_spectrum(res_rho)

#     k = kbins[0]
#     k_min, k_cutoff = k[0], k[int(0.2*len(k))]
#     k_weights = jnp.expand_dims(jnp.exp(-(k - k_min) / k_cutoff), 0)

#     cl_loss = jnp.sum(((res_cls/ref_cls - 1)*k_weights)**2, axis=-1)
#     cl_loss = jnp.mean(cl_loss)

#     return pos_loss + 0.1 * cl_loss

### field-level

In [13]:
# def field_loss_fn(model, i0, i1, architecture, eps=1e-8):
#     print("using voxel MSE")
    
#     res = solve_ode_sub(model, i0, i1, architecture)
#     gas_poss = res[2]%mesh_per_dim
    
#     rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)

#     ref_rho_sub = jnp.take(ref_rho, jnp.arange(i0, i1), axis=0)

#     loss = (rho - ref_rho_sub)**2

#     # the first snapshot is perfect by construction
#     loss = loss[1:]

#     # loss = loss[-1]
    
#     # loss /= jnp.maximum(scales.reshape(-1,1,1,1)**2, eps)
#     loss = jnp.mean(loss)
        
#     return loss

In [14]:
def field_loss_fn(model, y0, ts, ref, architecture, eps=1e-8):
    print("using voxel MSE")
    
    res = solve_ode_sub(model, y0, ts, architecture)
    gas_poss = res[2]%mesh_per_dim
    
    rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)

    loss = (rho - ref)**2

    # the first snapshot is perfect by construction
    loss = loss[1:]

    # loss = loss[-1]
    
    # loss /= jnp.maximum(scales.reshape(-1,1,1,1)**3, eps)
    loss /= jnp.maximum(scales.reshape(-1,1,1,1,1)**3, eps)

    loss = jnp.mean(loss)
        
    return loss

In [15]:
def batched_field_loss_fn(model, y0_batch, ts_batch, ref_batch, architecture, eps=1e-8):
    """Process multiple initial conditions in a batch"""
    def single_loss_fn(model, y0, ts, ref):
        return field_loss_fn(model, y0, ts, ref, architecture, eps=1e-8)
        
    batch_losses = jax.vmap(single_loss_fn, in_axes=(None,0,0,0))(model, y0_batch, ts_batch, ref_batch)
    
    return jnp.mean(batch_losses)

# architecture

### CNN

In [16]:
model = ScaleConditionedCNN(
    d_in=4, 
    d_out=1,
    d_hidden=128,
    # n_hidden=4,
    # d_hidden=256,
    n_hidden=4,
    kernel_size=(3, 3, 3),
    rngs=nnx.Rngs(0),
    norm_type="layer",
    activation=jax.nn.swish,
)

architecture = "cnn"

# training

In [17]:
# delta_snapshot = 2
delta_snapshot = 8

total_steps = 200
learning_rate = 1e-4
# learning_rate = 1e-5
# learning_rate = 1e-6
# learning_rate = optax.cosine_decay_schedule(
#     init_value=1e-4, 
#     decay_steps=total_steps, 
#     alpha=0.01,
# )
clip_norm = 1.0

optimizer = nnx.Optimizer(
    model,
    optax.chain(
        optax.clip_by_global_norm(clip_norm),
        # optax.ema(decay=0.999),
        # optax.adamw(learning_rate, b1=0.95, b2= 0.9999, eps=1e-5)
        optax.adam(learning_rate)
    )
)

losses = []
grad_norms = []

### single example

In [ ]:
for i in (pbar := tqdm.tqdm(range(total_steps))):
    i_rand = np.random.randint(0, len(scales) - delta_snapshot)
    i0, i1 = i_rand, i_rand + delta_snapshot
    
    y0 = (dm_poss[i0], dm_vels[i0], gas_poss[i0], gas_vels[i0])
    ts = scales[i0:i1]
    ref = ref_rho[i0:i1]
    
    loss, grad_norm = train_step(model, optimizer, y0, ts, ref)

    losses.append(float(loss))
    grad_norms.append(float(grad_norm))
    pbar.set_description(f"[{i0},{i1}], Loss: {loss:.4e}, Grad norm: {grad_norm:.4e}")

fig, ax = plt.subplots(figsize=(6,10), nrows=2, sharex=True)
ax[0].plot(losses)
ax[0].set(yscale="log", title="loss")

ax[1].plot(grad_norms)
ax[1].set(yscale="log", title="norm(grad)")

  0%|          | 0/200 [00:00<?, ?it/s]

using voxel MSE
dark matter and gas
Using CNN forces
Using learned pressure force
No latent variable
dark matter and gas
Using CNN forces
Using learned pressure force
No latent variable
dark matter and gas
Using CNN forces
Using learned pressure force
No latent variable


[12,20], Loss: 8.3377e-01, Grad norm: 9.1266e-01:   5%|▌         | 10/200 [00:30<05:53,  1.86s/it]

### batched

In [ ]:
batch_size = 2

In [ ]:
# def create_batch(batch_size):
#     y0_batch = []
#     ts_batch = []
#     ref_batch = []
    
#     for _ in range(batch_size):
#         i_rand = np.random.randint(0, len(scales) - delta_snapshot)
#         i0, i1 = i_rand, i_rand + delta_snapshot
        
#         y0 = (dm_poss[i0], dm_vels[i0], gas_poss[i0], gas_vels[i0])
#         ts = scales[i0:i1]
#         ref = ref_rho[i0:i1]
        
#         y0_batch.append(y0)
#         ts_batch.append(ts)
#         ref_batch.append(ref)

#     y0_batch = jax.tree_util.tree_map(
#         lambda *xs: jnp.stack(xs), 
#         *y0_batch
#     )
#     ts_batch = jnp.stack(ts_batch, axis=0)
#     ref_batch = jnp.stack(ref_batch, axis=0)

#     return i0, i1, y0_batch, ts_batch, ref_batch


In [ ]:
# for i in (pbar := tqdm.tqdm(range(total_steps))):
#     i0, i1, y0_batch, ts_batch, ref_batch = create_batch(batch_size)
    
#     loss, grad_norm = batched_train_step(model, optimizer, y0_batch, ts_batch, ref_batch)

#     losses.append(float(loss))
#     grad_norms.append(float(grad_norm))
#     pbar.set_description(f"[{i0},{i1}], Loss: {loss:.4e}, Grad norm: {grad_norm:.4e}")
    
# fig, ax = plt.subplots(figsize=(6,10), nrows=2, sharex=True)
# ax[0].plot(losses)
# ax[0].set(yscale="log", title="loss")

# ax[1].plot(grad_norms)
# ax[1].set(yscale="log", title="norm(grad)")

# run the simulation

In [ ]:
diagnostics.run_simulations(
    train_dict, 
    mesh_per_dim, 
    pressure_model=model, 
    gas_architecture=architecture, 
    dt0=dt0, 
    # i_snapshots=i_snapshots,
    # i_snapshots=np.arange(0,4),
    i_snapshots=np.arange(0,8),
    # i_snapshots=np.arange(5,7),
    # i_snapshots=np.arange(10,14),
    # i_snapshots=np.arange(-12,0),
    # i_snapshots=np.arange(-8,0),
)

In [ ]:
stop

In [ ]:
vali_dict = camels.load_CV_snapshots(
    "CV_1",
    mesh_per_dim,
    parts_per_dim,
    return_hydro=True,
)

In [ ]:
diagnostics.run_simulations(
    vali_dict, 
    mesh_per_dim, 
    pressure_model=model, 
    gas_architecture=architecture, 
    dt0=dt0, 
    # i_snapshots=i_snapshots,
    # i_snapshots=np.arange(0,4),
    i_snapshots=np.arange(0,8),
    # i_snapshots=np.arange(5,7),
    # i_snapshots=np.arange(10,14),
    # i_snapshots=np.arange(-12,0),
    # i_snapshots=np.arange(-8,0),
)